# Continuous Batching

**Tags:** inference, optimization
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/continuous-batching.md

## TL;DR

Serve LLM requests dynamically: new requests join in-flight batch immediately, don't wait for full batch. Reduces latency 5-20x compared to static batching; same throughput. Enables real-time serving at high load. Trade: complex KV cache management, memory fragmentation.

## Core Intuition

Traditional batching: wait for 64 requests to arrive, then process together (all latency = slowest request). Continuous batching: start with 1 request, add 2 at 10ms, add 3 at 20ms... Each request processes as soon as it arrives, finished requests leave, new ones join. Result: early requests finish in 10-100ms instead of waiting 1000ms.

## How It Works

**Static Batching (Request-Level Batching):**
```
Incoming requests arrive over time:
Req1: 0ms (waits)
Req2: 50ms (waits)
...
Req64: 1000ms (last request)

At 1000ms: combine all 64 → compute together
All requests finish at ~1050ms (slowest latency dominates)

Pros: simple, high throughput
Cons: high latency (especially early requests), wasted compute waiting
```

**Continuous Batching (Token-Level Batching):**
```
Req1 arrives 0ms: starts generating
  Gen 1: compute 1 token for Req1 (1ms)
  
Req2 arrives 10ms: joins in-flight computation
  Gen 2: compute 1 token for Req1, Req2 (2ms batched)
  
Req3 arrives 20ms: joins
  Gen 3: compute 1 token for Req1, Req2, Req3 (3ms batched)
  
Req1 finishes at 100ms: leaves batch
  Gen 4: compute 1 token for Req2, Req3 (2ms batched)
  
Req2 finishes at 150ms: leaves batch
  Gen 5: compute 1 token for Req3 (1ms)
  
Req3 finishes at 200ms: done

Result: streaming latencies (100ms, 150ms, 200ms) instead of all waiting 1000ms
```

**Implementation Details:**

```
State Tracking:
- Each request: (request_id, input_tokens, generated_tokens, seq_len)
- Shared KV cache: (batch_size, seq_len, hidden_dim)
- Active batch: set of requests currently processing

Algorithm:
1. Scheduler loop (every 5-10ms):
   a. Check new requests from queue
   b. Append to active batch
   c. Run forward pass on all active sequences
   d. Generate next token for each
   e. Remove finished sequences (reached max_tokens)
   f. Loop back to step 1

Challenges:
- KV cache indexing: requests at different positions in batch
  Solution: flatten to (total_tokens, hidden_dim), track offsets
- Different sequence lengths: can't use standard matrix ops
  Solution: packed sequences or ragged tensors
- Memory fragmentation: removing finished requests leaves gaps
  Solution: periodic defragmentation or memory pools
```

**KV Cache Management:**

Standard (Static Batching):
```
Batch size: 64
Max seq_len: 2048
KV cache shape: (64, 2048, 128)  # 128 = num_heads × head_dim
Memory: 64 × 2048 × 128 × 2 (K and V) × 4 bytes = 67 MB (wasteful if seqs short)
```

Continuous (Packed Format):
```
Req1: pos 0-50
Req2: pos 50-100
Req3: pos 100-150

KV cache shape: (total_tokens, hidden_dim) = (150, 128)
Memory: 150 × 128 × 2 × 4 bytes = 154 KB (much smaller!)
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Metric | Static | Continuous | Notes |
|--------|--------|-----------|-------|
| Latency (p50) | 500ms | 50ms | 10x improvement |
| Latency (p99) | 1000ms | 200ms | Heavy requests less impact |
| Throughput | 64 req/s (60s batch) | 64 req/s (10ms batches) | Same overall |
| Latency per token | 1ms | 1-2ms | Continuous slightly higher (overhead) |
| Memory | 67 MB (static batch) | 5-10 MB (active) | 10x less peak memory |
| Implementation | Simple | Complex | vLLM, DeepSpeed handle this |

**Real-world latency comparison:**

Scenario: serving 100 QPS, each request generates 100 tokens

Static batching (batch=64, take every second):
```
Batch 1: 64 requests, takes 150ms total
  Req 1: latency = 150ms
  Req 64: latency = 150ms
Batch 2: 36 requests (partial), takes 85ms
  Req 65: latency = 150 + 85 = 235ms
  Req 100: latency = 235ms

Average latency: ~190ms
```

Continuous batching:
```
Req 1: 0ms → 50ms (latency = 50ms, finished)
Req 2: 10ms → 60ms (latency = 50ms)
...
Req 100: 990ms → 1040ms (latency = 50ms)

Average latency: ~50ms (consistent, 4x better)
```

### Code Implementation

```python
# Key imports for Continuous Batching
import numpy as np
import torch
from typing import Any

# Continuous Batching example implementation
class ContinuousBatching:
    """
    Continuous Batching implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is Continuous Batching used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Continuous Batching?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Continuous Batching compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Continuous Batching?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Continuous Batching?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/continuous-batching.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]